# 📊 Insurance Claim NLP — Exploratory Data Analysis

This notebook walks through the dataset used in the project:
- Dataset shape and column overview
- Label distribution (Coverage Code)
- Accident Source breakdown
- Claim description text statistics
- Word frequency analysis
- NLP preprocessing demo
- TF-IDF feature inspection
- Model performance comparison

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter

# Display settings
pd.set_option('display.max_colwidth', 80)
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100

print('Libraries loaded ✓')

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../artifacts/insurance_claims_data.csv')
print(f'Shape: {df.shape}')
df.head(10)

In [ ]:
print('Column names:', df.columns.tolist())
print('\nData types:')
print(df.dtypes)
print('\nNull values:')
print(df.isnull().sum())

## 2. Target Label Distribution (Coverage Code)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
counts = df['Coverage Code'].value_counts()
axes[0].bar(counts.index, counts.values,
            color=sns.color_palette('Set2', len(counts)))
axes[0].set_title('Coverage Code — Frequency', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Claims')
for bar, val in zip(axes[0].patches, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.3, str(val), ha='center', fontsize=10)

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('Set2', len(counts)),
            startangle=90, pctdistance=0.82)
axes[1].set_title('Coverage Code — Proportion', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()
print(counts)

## 3. Accident Source Breakdown

In [ ]:
src_counts = df['Accident Source'].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('pastel', len(src_counts))
ax.barh(src_counts.index[::-1], src_counts.values[::-1], color=colors[::-1])
ax.set_title('Accident Source Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Claims')
for i, val in enumerate(src_counts.values[::-1]):
    ax.text(val + 0.05, i, str(val), va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 4. Coverage Code vs Accident Source (Heatmap)

In [ ]:
cross = pd.crosstab(df['Coverage Code'], df['Accident Source'])

plt.figure(figsize=(14, 5))
sns.heatmap(cross, annot=True, fmt='d', cmap='Blues',
            linewidths=0.5, linecolor='white')
plt.title('Coverage Code vs Accident Source', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Claim Description — Text Statistics

In [ ]:
df['word_count'] = df['Claim Description'].str.split().str.len()
df['char_count'] = df['Claim Description'].str.len()

print('Word count statistics:')
print(df['word_count'].describe().round(2))
print('\nCharacter count statistics:')
print(df['char_count'].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['word_count'], bins=15, color='#42a5f5', edgecolor='white')
axes[0].set_title('Word Count per Claim', fontweight='bold')
axes[0].set_xlabel('Number of Words')
axes[0].set_ylabel('Frequency')

axes[1].hist(df['char_count'], bins=15, color='#66bb6a', edgecolor='white')
axes[1].set_title('Character Count per Claim', fontweight='bold')
axes[1].set_xlabel('Number of Characters')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 6. Most Frequent Words (Raw)

In [ ]:
all_text = ' '.join(df['Claim Description'].str.lower())
all_words = re.sub(r'[^a-z\s]', ' ', all_text).split()

STOP_WORDS = {
    'a','an','the','and','or','but','in','on','at','to','for','of','with',
    'by','from','is','was','were','are','be','been','has','have','had',
    'it','its','this','that','these','those','during','causing','caused'
}
filtered_words = [w for w in all_words if w not in STOP_WORDS and len(w) > 2]

word_freq = Counter(filtered_words)
top_words = pd.Series(dict(word_freq.most_common(25)))

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('Blues_r', len(top_words))
ax.barh(top_words.index[::-1], top_words.values[::-1], color=colors)
ax.set_title('Top 25 Most Frequent Words in Claim Descriptions', fontsize=13, fontweight='bold')
ax.set_xlabel('Frequency')
plt.tight_layout()
plt.show()

## 7. NLP Preprocessing Demo

In [ ]:
import sys
sys.path.append('..')
from src.components.data_transformation import clean_text

sample_claims = df['Claim Description'].head(5).tolist()

print('Before & After Cleaning:\n')
for i, claim in enumerate(sample_claims):
    print(f'  [{i+1}] ORIGINAL : {claim}')
    print(f'      CLEANED  : {clean_text(claim)}')
    print()

## 8. TF-IDF Feature Inspection

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

cleaned = df['Claim Description'].apply(clean_text)

tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=5000, sublinear_tf=True)
X = tfidf.fit_transform(cleaned)

print(f'TF-IDF matrix shape: {X.shape}')
print(f'Total features (unigrams + bigrams): {len(tfidf.get_feature_names_out())}')
print(f'\nSample features:')
print(tfidf.get_feature_names_out()[:30])

## 9. Top TF-IDF Features per Coverage Code

In [ ]:
import numpy as np

feature_names = tfidf.get_feature_names_out()
labels = df['Coverage Code'].values

fig, axes = plt.subplots(1, len(df['Coverage Code'].unique()),
                         figsize=(16, 5), sharey=False)

for ax, label in zip(axes, sorted(df['Coverage Code'].unique())):
    mask = labels == label
    mean_tfidf = X[mask].mean(axis=0).A1
    top_idx = mean_tfidf.argsort()[-10:][::-1]
    top_features = [feature_names[i] for i in top_idx]
    top_scores   = mean_tfidf[top_idx]

    ax.barh(top_features[::-1], top_scores[::-1],
            color=sns.color_palette('Set2', 1)[0])
    ax.set_title(f'{label}', fontweight='bold', fontsize=11)
    ax.set_xlabel('Mean TF-IDF')

plt.suptitle('Top 10 TF-IDF Terms per Coverage Code', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 10. Model Comparison (Quick Run)

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

le = LabelEncoder()
y = le.fit_transform(df['Coverage Code'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=5.0),
    'Linear SVM'         : LinearSVC(C=1.0, max_iter=2000),
    'Multinomial NB'     : MultinomialNB(alpha=0.1),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42),
    'SGD Classifier'     : SGDClassifier(loss='modified_huber', random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    results[name] = acc
    print(f'  {name:<25} → {acc:.4f}')

best = max(results, key=results.get)
print(f'\n  🏆 Best: {best} ({results[best]:.2%})')

In [ ]:
# Plot model comparison
results_s = pd.Series(results).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#43a047' if v == results_s.max() else '#90caf9' for v in results_s.values]
ax.barh(results_s.index, results_s.values, color=colors)
ax.set_xlim(0, 1.05)
ax.set_xlabel('Test Accuracy')
ax.set_title('Model Comparison — Test Accuracy', fontsize=13, fontweight='bold')
for i, val in enumerate(results_s.values):
    ax.text(val + 0.01, i, f'{val:.2%}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

## 11. Classification Report (Best Model)


In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix

best_model = models[best]
y_pred = best_model.predict(X_test)

print(f'Best model: {best}\n')
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_,
            linewidths=0.5)
ax.set_title(f'Confusion Matrix — {best}', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## 12. Try a Manual Prediction

Type your own claim text and see what the model predicts.

In [ ]:
test_claims = [
    "Worker fell from ladder and broke his arm at construction site",
    "Car rear-ended at traffic lights causing neck injury",
    "Customer slipped on wet floor in supermarket",
    "House fire caused by faulty wiring in living room",
    "Employee developed back pain from lifting heavy packages",
]

for claim in test_claims:
    cleaned = clean_text(claim)
    x_vec   = tfidf.transform([cleaned])
    pred    = le.inverse_transform(best_model.predict(x_vec))[0]
    print(f'  Claim   : {claim}')
    print(f'  → Predicted Coverage Code: {pred}\n')